# 03 Run Experiments

            Objective: run both tasks for each locally provided model, cache every raw response, and preserve invalid outputs for auditability.

            This notebook is intentionally guarded. Set `RUN_FULL_EXPERIMENT = True` only after the pilot passes.


In [ ]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
while not (PROJECT_ROOT / "AGENTS.md").exists() and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT / "scripts"))
import eval_utils as eu

CONFIG_PATH = PROJECT_ROOT / "config.json"
if not CONFIG_PATH.exists():
    CONFIG_PATH = PROJECT_ROOT / "config.example.json"
CONFIG = eu.load_config(CONFIG_PATH)
eu.ensure_project_dirs(PROJECT_ROOT)

PROJECT_ROOT, CONFIG_PATH


## Configure Run


In [ ]:
HOST = os.getenv("HOST", CONFIG["llm"]["host"])
            MODELS = [m.strip() for m in os.getenv("MODELS", ",".join(CONFIG["llm"]["models"])).split(",") if m.strip()]
            RUN_FULL_EXPERIMENT = os.getenv("RUN_FULL_EXPERIMENT", "false").lower() in {"1", "true", "yes"}
            deterministic = CONFIG["llm"]["deterministic"]
            stochastic = CONFIG["llm"]["stochastic"]
            output_path = PROJECT_ROOT / "data/processed/model_outputs_raw.jsonl"
            run_id = eu.new_run_id("full")

            benchmark = eu.read_csv_rows(PROJECT_ROOT / "data/processed/benchmark_items.csv")
            planned_calls = len(benchmark) * len(MODELS) * 2 * (1 + int(stochastic["samples"]))
            print({"HOST": HOST, "MODELS": MODELS, "RUN_FULL_EXPERIMENT": RUN_FULL_EXPERIMENT, "run_id": run_id})
            print(f"Benchmark items: {len(benchmark)}")
            print(f"Planned calls: {planned_calls}")


## Run Full Experiment


In [ ]:
task1_template = eu.load_prompt(PROJECT_ROOT / "prompts/mandatory_entailment.txt")
task2_template = eu.load_prompt(PROJECT_ROOT / "prompts/modality_extraction.txt")

def prompt_for(task, item):
    if task == "task1":
        return eu.render_prompt(
            task1_template,
            source_statement=item["source_statement"],
            candidate_requirement=item["candidate_requirement"],
        )
    if task == "task2":
        return eu.render_prompt(task2_template, source_statement=item["source_statement"])
    raise ValueError(task)

def run_one(item, task, model, sample_kind, sample_index, temperature, top_p, output_path, run_id):
    prompt = prompt_for(task, item)
    completion = eu.chat_completion(
        host=HOST,
        model=model,
        prompt=prompt,
        temperature=temperature,
        top_p=top_p,
        max_tokens=int(CONFIG["llm"]["max_tokens"]),
        timeout_s=int(CONFIG["llm"]["timeout_s"]),
        api_key_env=CONFIG["llm"]["api_key_env"],
    )
    record = eu.build_raw_record(
        run_id=run_id,
        model=model,
        host=HOST,
        task=task,
        item=item,
        sample_index=sample_index,
        sample_kind=sample_kind,
        temperature=temperature,
        top_p=top_p,
        prompt_version=CONFIG["project"]["prompt_version"],
        prompt=prompt,
        completion=completion,
    )
    eu.append_jsonl(output_path, record)
    return record


In [ ]:
if RUN_FULL_EXPERIMENT:
                total = 0
                for model in MODELS:
                    print(f"Running model: {model}")
                    for item in benchmark:
                        for task in ["task1", "task2"]:
                            run_one(
                                item=item,
                                task=task,
                                model=model,
                                sample_kind="deterministic",
                                sample_index=0,
                                temperature=float(deterministic["temperature"]),
                                top_p=float(deterministic["top_p"]),
                                output_path=output_path,
                                run_id=run_id,
                            )
                            total += 1
                            for sample_index in range(int(stochastic["samples"])):
                                run_one(
                                    item=item,
                                    task=task,
                                    model=model,
                                    sample_kind="stochastic",
                                    sample_index=sample_index,
                                    temperature=float(stochastic["temperature"]),
                                    top_p=float(stochastic["top_p"]),
                                    output_path=output_path,
                                    run_id=run_id,
                                )
                                total += 1
                            if total % 100 == 0:
                                print(f"Completed {total}/{planned_calls} calls")
                print(f"Done. Wrote records to {output_path}")
            else:
                print("Full experiment not run. Set RUN_FULL_EXPERIMENT=true after the pilot gate passes.")


## Parse-Failure Audit


In [ ]:
all_rows = eu.read_jsonl(output_path)
            run_rows = [row for row in all_rows if row.get("run_id") == run_id]
            if run_rows:
                status_counts = {}
                for row in run_rows:
                    status_counts[row["parse_status"]] = status_counts.get(row["parse_status"], 0) + 1
                print(status_counts)
                print(f"Parse success rate: {status_counts.get('ok', 0) / len(run_rows):.3f}")
            else:
                print("No rows for this run_id yet.")
